# Chinese → Vietnamese Translation Baseline

This notebook builds an end-to-end machine translation pipeline:

- load and split the data
- train SentencePiece tokenizers
- train multiple Seq2Seq models
- ensemble inference with token voting
- export `submission.csv` and `submission.zip`

The code is written to be easy to debug cell by cell.

In [1]:
# %pip install -q torch pandas sentencepiece sacrebleu tqdm

## 1) Imports and configuration

In [2]:
import os
import random
import zipfile
from pathlib import Path
from collections import Counter

import pandas as pd
import sentencepiece as spm
import sacrebleu
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# config
def find_existing(*candidates):
    for p in candidates:
        if p.exists():
            return p
    return None

TRAIN_SRC = find_existing(
    Path("dataset/train/train.zh"),
    Path("/mnt/data/train.zh"),
    Path("train.zh"),
)
TRAIN_TGT = find_existing(
    Path("dataset/train/train.vi"),
    Path("/mnt/data/train.vi"),
    Path("train.vi"),
)
TEST_SRC = find_existing(
    Path("dataset/test/test.zh"),
    Path("/mnt/data/test.zh"),
    Path("test.zh"),
)

assert TRAIN_SRC is not None, "Cannot find train.zh"
assert TRAIN_TGT is not None, "Cannot find train.vi"

print("TRAIN_SRC:", TRAIN_SRC)
print("TRAIN_TGT:", TRAIN_TGT)
print("TEST_SRC :", TEST_SRC)

SAVE_DIR = Path("./checkpoints_mt")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

VOCAB_SIZE = 8000
EMB_SIZE = 256
HID_SIZE = 512
NUM_LAYERS = 1
DROPOUT = 0.25
MAX_LEN = 80

BATCH_SIZE = 64
EPOCHS = 36
LR = 2e-4
WEIGHT_DECAY = 1e-5
CLIP_GRAD = 1.0
VALID_RATIO = 0.1

N_MODELS = 3
ENSEMBLE_SEEDS = [42, 36, 72]

BASE_SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PAD_ID = 0
UNK_ID = 1
BOS_ID = 2
EOS_ID = 3

def set_seed(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if torch.cuda.is_available():
        torch.backends.cudnn.deterministic = False
        torch.backends.cudnn.benchmark = True

set_seed(BASE_SEED)
print("DEVICE:", DEVICE)

PyTorch: 2.11.0+cu130
CUDA available: True
TRAIN_SRC: dataset/train/train.zh
TRAIN_TGT: dataset/train/train.vi
TEST_SRC : dataset/test/test.zh
DEVICE: cuda


/home/izu/Projects/olpai/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2) Load and sanity-check data

In [3]:
def read_lines(path: Path):
    with path.open("r", encoding="utf-8") as f:
        return [line.strip() for line in f if line.strip()]

train_src = read_lines(TRAIN_SRC)
train_tgt = read_lines(TRAIN_TGT)

assert len(train_src) == len(train_tgt), f"Mismatch: {len(train_src)} vs {len(train_tgt)}"

print("Training pairs:", len(train_src))
print("Example source:", train_src[0])
print("Example target:", train_tgt[0])

if TEST_SRC is not None and TEST_SRC.exists():
    test_src = read_lines(TEST_SRC)
    print("Test samples:", len(test_src))
    print("Example test:", test_src[0])
else:
    test_src = None
    print("No test.zh found yet. Add it before running inference.")

Training pairs: 25648
Example source: 这个 巴士 去 联合 广场 的 度假 旅馆 吗 ？
Example target: Xe_buýt này có đi đến Quảng_trường Holiday_Inn_Union không ?
Test samples: 6413
Example test: 还 早 呀 ， 过 几 年 再 说 。


## 3) Shuffle and split train/validation

In [4]:
indices = list(range(len(train_src)))
rng = random.Random(BASE_SEED)
rng.shuffle(indices)

shuffled_src = [train_src[i] for i in indices]
shuffled_tgt = [train_tgt[i] for i in indices]

split_idx = int(len(shuffled_src) * (1 - VALID_RATIO))
train_src_split = shuffled_src[:split_idx]
train_tgt_split = shuffled_tgt[:split_idx]
valid_src = shuffled_src[split_idx:]
valid_tgt = shuffled_tgt[split_idx:]

print("Train split:", len(train_src_split))
print("Valid split:", len(valid_src))

Train split: 23083
Valid split: 2565


## 4) Train or load SentencePiece tokenizers

In [5]:
spm_dir = SAVE_DIR / "spm"
spm_dir.mkdir(parents=True, exist_ok=True)

ZH_PREFIX = spm_dir / "zh_bpe"
VI_PREFIX = spm_dir / "vi_bpe"

def train_sentencepiece(texts, model_prefix: Path, vocab_size: int):
    tmp_file = model_prefix.with_suffix(".txt")
    with tmp_file.open("w", encoding="utf-8") as f:
        for line in texts:
            f.write(line + "\n")
    spm.SentencePieceTrainer.Train(
        input=str(tmp_file),
        model_prefix=str(model_prefix),
        vocab_size=vocab_size,
        model_type="bpe",
        character_coverage=1.0,
        pad_id=PAD_ID,
        unk_id=UNK_ID,
        bos_id=BOS_ID,
        eos_id=EOS_ID,
    )
    print("Trained:", model_prefix)

def load_sp(model_path: Path):
    sp = spm.SentencePieceProcessor()
    sp.Load(str(model_path))
    return sp

if not ZH_PREFIX.with_suffix(".model").exists():
    train_sentencepiece(train_src_split, ZH_PREFIX, VOCAB_SIZE)
else:
    print("Loaded existing Chinese tokenizer.")

if not VI_PREFIX.with_suffix(".model").exists():
    train_sentencepiece(train_tgt_split, VI_PREFIX, VOCAB_SIZE)
else:
    print("Loaded existing Vietnamese tokenizer.")

sp_zh = load_sp(ZH_PREFIX.with_suffix(".model"))
sp_vi = load_sp(VI_PREFIX.with_suffix(".model"))

SRC_VOCAB = sp_zh.GetPieceSize()
TGT_VOCAB = sp_vi.GetPieceSize()

print("Source vocab:", SRC_VOCAB)
print("Target vocab:", TGT_VOCAB)

def encode_src(text: str, sp):
    ids = sp.EncodeAsIds(text)
    return [BOS_ID] + ids[:MAX_LEN - 2] + [EOS_ID]

def encode_tgt(text: str, sp):
    ids = sp.EncodeAsIds(text)
    return [BOS_ID] + ids[:MAX_LEN - 2] + [EOS_ID]

Loaded existing Chinese tokenizer.
Loaded existing Vietnamese tokenizer.
Source vocab: 4000
Target vocab: 4000


## 5) Dataset and dataloader

In [6]:
class TranslationDataset(Dataset):
    def __init__(self, src_texts, tgt_texts, sp_src, sp_tgt):
        self.src_texts = src_texts
        self.tgt_texts = tgt_texts
        self.sp_src = sp_src
        self.sp_tgt = sp_tgt

    def __len__(self):
        return len(self.src_texts)

    def __getitem__(self, idx):
        src_ids = encode_src(self.src_texts[idx], self.sp_src)
        tgt_ids = encode_tgt(self.tgt_texts[idx], self.sp_tgt)
        return torch.tensor(src_ids, dtype=torch.long), torch.tensor(tgt_ids, dtype=torch.long)

def collate_fn(batch):
    srcs, tgts = zip(*batch)
    max_src = max(len(x) for x in srcs)
    max_tgt = max(len(x) for x in tgts)

    src_pad = torch.full((len(batch), max_src), PAD_ID, dtype=torch.long)
    tgt_pad = torch.full((len(batch), max_tgt), PAD_ID, dtype=torch.long)

    for i, (s, t) in enumerate(zip(srcs, tgts)):
        src_pad[i, :len(s)] = s
        tgt_pad[i, :len(t)] = t

    return src_pad, tgt_pad

train_dataset = TranslationDataset(train_src_split, train_tgt_split, sp_zh, sp_vi)
valid_dataset = TranslationDataset(valid_src, valid_tgt, sp_zh, sp_vi)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

print("Train batches:", len(train_loader))
print("Valid batches:", len(valid_loader))

Train batches: 361
Valid batches: 41


## 6) Seq2Seq model with attention

In [7]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, emb_size, hidden_size, num_layers=1, dropout=0.25):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_size, padding_idx=PAD_ID)
        self.dropout = nn.Dropout(dropout)
        self.rnn = nn.GRU(
            emb_size,
            hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
        )
        self.fc_hidden = nn.Linear(hidden_size * 2, hidden_size)

    def forward(self, src):
        emb = self.dropout(self.embedding(src))
        outputs, hidden = self.rnn(emb)
        h = outputs.size(-1) // 2
        outputs = outputs[:, :, :h] + outputs[:, :, h:]

        # Use final layer's forward/backward hidden states
        if hidden.size(0) >= 2:
            hidden_cat = torch.cat([hidden[-2], hidden[-1]], dim=1)
            hidden = torch.tanh(self.fc_hidden(hidden_cat)).unsqueeze(0)
        else:
            hidden = torch.tanh(self.fc_hidden(torch.cat([hidden[-1], hidden[-1]], dim=1))).unsqueeze(0)
        return outputs, hidden

class Attention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size * 2, hidden_size)
        self.v = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        src_len = encoder_outputs.size(1)
        hidden_rep = hidden[-1].unsqueeze(1).repeat(1, src_len, 1)
        energy = torch.tanh(self.attn(torch.cat((hidden_rep, encoder_outputs), dim=2)))
        scores = self.v(energy).squeeze(2)
        return torch.softmax(scores, dim=1)

class Decoder(nn.Module):
    def __init__(self, vocab_size, emb_size, hidden_size, dropout=0.25):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_size, padding_idx=PAD_ID)
        self.dropout = nn.Dropout(dropout)
        self.attention = Attention(hidden_size)
        self.rnn = nn.GRU(emb_size + hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size * 2, vocab_size)

    def forward(self, input_step, hidden, encoder_outputs):
        emb = self.dropout(self.embedding(input_step))
        attn_weights = self.attention(hidden, encoder_outputs).unsqueeze(1)
        context = torch.bmm(attn_weights, encoder_outputs)
        rnn_input = torch.cat([emb, context], dim=2)
        output, hidden = self.rnn(rnn_input, hidden)
        pred = self.fc(torch.cat([output.squeeze(1), context.squeeze(1)], dim=1))
        return pred, hidden

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size, tgt_len = tgt.size(0), tgt.size(1)
        vocab_size = self.decoder.fc.out_features
        outputs = torch.zeros(batch_size, tgt_len, vocab_size, device=src.device)

        encoder_outputs, hidden = self.encoder(src)
        input_step = tgt[:, 0].unsqueeze(1)

        for t in range(1, tgt_len):
            pred, hidden = self.decoder(input_step, hidden, encoder_outputs)
            outputs[:, t] = pred

            use_teacher = random.random() < teacher_forcing_ratio
            next_token = tgt[:, t].unsqueeze(1) if use_teacher else pred.argmax(dim=1).unsqueeze(1)
            input_step = next_token

        return outputs

## 7) Training and evaluation helpers

In [8]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

@torch.no_grad()
def greedy_translate_batch(model, src, max_len=MAX_LEN):
    model.eval()
    encoder_outputs, hidden = model.encoder(src)
    input_step = torch.full((src.size(0), 1), BOS_ID, dtype=torch.long, device=src.device)

    preds = [[] for _ in range(src.size(0))]
    finished = [False] * src.size(0)

    for _ in range(max_len):
        logits, hidden = model.decoder(input_step, hidden, encoder_outputs)
        next_tokens = logits.argmax(dim=1)
        input_step = next_tokens.unsqueeze(1)

        for i, tok in enumerate(next_tokens.tolist()):
            if not finished[i]:
                if tok == EOS_ID:
                    finished[i] = True
                else:
                    preds[i].append(tok)
    return preds

@torch.no_grad()
def evaluate_bleu(model, loader, sp_tgt):
    model.eval()
    hyps, refs = [], []

    for src, tgt in tqdm(loader, desc="Validation", leave=False):
        src = src.to(DEVICE)
        pred_ids = greedy_translate_batch(model, src, max_len=MAX_LEN)

        for i in range(len(pred_ids)):
            hyp = sp_tgt.DecodeIds(pred_ids[i])
            ref_ids = tgt[i].tolist()
            ref_ids = [x for x in ref_ids if x not in (PAD_ID, BOS_ID, EOS_ID)]
            ref = sp_tgt.DecodeIds(ref_ids)
            hyps.append(hyp)
            refs.append(ref)

    bleu = sacrebleu.corpus_bleu(hyps, [refs])
    return bleu.score

def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss = 0.0

    pbar = tqdm(loader, desc="Train", leave=False)
    for src, tgt in pbar:
        src = src.to(DEVICE)
        tgt = tgt.to(DEVICE)

        optimizer.zero_grad(set_to_none=True)
        output = model(src, tgt, teacher_forcing_ratio=0.5)

        loss = criterion(
            output[:, 1:].reshape(-1, output.size(-1)),
            tgt[:, 1:].reshape(-1),
        )
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CLIP_GRAD)
        optimizer.step()

        total_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return total_loss / max(1, len(loader))

def build_model():
    enc = Encoder(SRC_VOCAB, EMB_SIZE, HID_SIZE, NUM_LAYERS, DROPOUT)
    dec = Decoder(TGT_VOCAB, EMB_SIZE, HID_SIZE, DROPOUT)
    return Seq2Seq(enc, dec).to(DEVICE)

## 8) Train multiple models

In [9]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

ensemble_checkpoints = []
history = []

for model_idx, seed in enumerate(ENSEMBLE_SEEDS[:N_MODELS], start=1):
    print()
    print("=" * 70)
    print(f"Training model {model_idx}/{N_MODELS} | seed={seed}")
    print("=" * 70)
    set_seed(seed)

    model = build_model()
    print("Parameters:", f"{count_parameters(model):,}")

    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    best_bleu = -1.0
    best_path = SAVE_DIR / f"best_model_seed{seed}.pt"
    best_epoch = -1

    for epoch in range(1, EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer)
        val_bleu = evaluate_bleu(model, valid_loader, sp_vi)

        msg = f"seed={seed} epoch={epoch:02d} loss={train_loss:.4f} bleu={val_bleu:.2f}"
        if val_bleu > best_bleu:
            best_bleu = val_bleu
            best_epoch = epoch
            torch.save(
                {
                    "seed": seed,
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_bleu": best_bleu,
                    "config": {
                        "VOCAB_SIZE": VOCAB_SIZE,
                        "EMB_SIZE": EMB_SIZE,
                        "HID_SIZE": HID_SIZE,
                        "NUM_LAYERS": NUM_LAYERS,
                        "DROPOUT": DROPOUT,
                        "MAX_LEN": MAX_LEN,
                    },
                },
                best_path,
            )
            msg += "  <-- saved best"
        print(msg)

    print(f"Best for seed {seed}: BLEU={best_bleu:.2f} at epoch {best_epoch}")
    ensemble_checkpoints.append(best_path)
    history.append({"seed": seed, "best_bleu": best_bleu, "best_epoch": best_epoch, "path": str(best_path)})

history_df = pd.DataFrame(history)
history_df


Training model 1/3 | seed=42
Parameters: 11,532,704


That's 100 lines that end in a tokenized period ('.')                
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


seed=42 epoch=01 loss=5.1693 bleu=4.66  <-- saved best


That's 100 lines that end in a tokenized period ('.')                
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


seed=42 epoch=02 loss=4.3434 bleu=7.78  <-- saved best


...

That's 100 lines that end in a tokenized period ('.')                
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


seed=72 epoch=35 loss=0.8972 bleu=28.86


That's 100 lines that end in a tokenized period ('.')                
It looks like you forgot to detokenize your test data, which may hurt your score.
If you insist your data is detokenized, or don't care, you can suppress this message with the `force` parameter.


seed=72 epoch=36 loss=0.8776 bleu=28.95  <-- saved best
Best for seed 72: BLEU=28.95 at epoch 36


,seed,best_bleu,best_epoch,path
0,42,28.646738,34,checkpoints_mt/best_model_seed42.pt
1,36,29.061945,34,checkpoints_mt/best_model_seed36.pt
2,72,28.951588,36,checkpoints_mt/best_model_seed72.pt


## 9) Load ensemble models

In [10]:
def load_model_from_checkpoint(path: Path):
    ckpt = torch.load(path, map_location=DEVICE)
    model = build_model()
    model.load_state_dict(ckpt["model_state_dict"])
    model.to(DEVICE)
    model.eval()
    return model, ckpt

ensemble_models = []
for ckpt_path in ensemble_checkpoints:
    model, ckpt = load_model_from_checkpoint(ckpt_path)
    ensemble_models.append(model)
    print(f"Loaded {ckpt_path.name} | seed={ckpt['seed']} | best_bleu={ckpt['best_bleu']:.2f}")

print("Ensemble size:", len(ensemble_models))

Loaded best_model_seed42.pt | seed=42 | best_bleu=28.65
Loaded best_model_seed36.pt | seed=36 | best_bleu=29.06
Loaded best_model_seed72.pt | seed=72 | best_bleu=28.95
Ensemble size: 3


## 10) Ensemble voting during inference

In [11]:
@torch.no_grad()
def ensemble_vote_next_token(logits_list):
    # logits_list: list of [B, V] tensors
    probs_list = [torch.softmax(logits, dim=1) for logits in logits_list]
    top1_list = [probs.argmax(dim=1) for probs in probs_list]

    batch_size = logits_list[0].size(0)
    voted = []

    for b in range(batch_size):
        counts = Counter(int(top1[b].item()) for top1 in top1_list)
        top_count = max(counts.values())
        candidates = [tok for tok, cnt in counts.items() if cnt == top_count]

        if len(candidates) == 1:
            voted.append(candidates[0])
        else:
            scores = {}
            for tok in candidates:
                scores[tok] = sum(float(probs[b, tok].item()) for probs in probs_list)
            voted.append(max(scores, key=scores.get))

    return torch.tensor(voted, device=logits_list[0].device, dtype=torch.long)

@torch.no_grad()
def translate_with_ensemble(models, src_texts, sp_src, sp_tgt, max_len=MAX_LEN):
    results = []

    for text in tqdm(src_texts, desc="Translating"):
        src_ids = encode_src(text, sp_src)
        src = torch.tensor(src_ids, dtype=torch.long, device=DEVICE).unsqueeze(0)

        enc_outputs = []
        hiddens = []
        for model in models:
            eo, h = model.encoder(src)
            enc_outputs.append(eo)
            hiddens.append(h)

        inputs = [torch.full((1, 1), BOS_ID, dtype=torch.long, device=DEVICE) for _ in models]
        decoded = []

        for _ in range(max_len):
            logits_list = []
            next_hiddens = []

            for i, model in enumerate(models):
                logits, new_hidden = model.decoder(inputs[i], hiddens[i], enc_outputs[i])
                logits_list.append(logits)
                next_hiddens.append(new_hidden)

            voted_token = ensemble_vote_next_token(logits_list)
            token_id = int(voted_token.item())

            if token_id == EOS_ID:
                break

            decoded.append(token_id)

            for i in range(len(models)):
                inputs[i] = voted_token.view(1, 1)
                hiddens[i] = next_hiddens[i]

        results.append(sp_tgt.DecodeIds(decoded))

    return results

## 11) Run inference and create submission files

In [14]:
assert test_src is not None, "Add test.zh first, then run this cell."

predictions = translate_with_ensemble(ensemble_models, test_src, sp_zh, sp_vi, max_len=MAX_LEN)

submission = pd.DataFrame({
    "tieng_trung": test_src,
    "tieng_viet": predictions,
})

submission_csv = Path("submission.csv")
submission_zip = Path("submission.zip")

submission.to_csv(submission_csv, index=False, encoding="utf-8-sig")

with zipfile.ZipFile(submission_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(submission_csv, arcname="submission.csv")

print("Saved:", submission_csv.resolve())
print("Saved:", submission_zip.resolve())
submission.head(10)

Translating: 100%|██████████| 6413/6413 [03:40<00:00, 29.03it/s]

Saved: /home/izu/Projects/olpai/trans/submission.csv
Saved: /home/izu/Projects/olpai/trans/submission.zip


,tieng_trung,tieng_viet
0,还 早 呀 ， 过 几 年 再 说 。,"còn sớm mà , qua mấy năm rồi rồi tính nữa ."
1,你 跟 我 去吧,bạn đi tôi tôi nhe
2,我 都 喜欢 。,Tôi thích nó .
3,我 受伤 了 需要 一 辆 救护车 。,Tôi bị bị_thương một chiếc xe_cứu_thương .
4,等 一下 。 这个 是 吗 ？,Chờ . cái này phải không ?
5,我 想 烫 一下 这 套 衣服,tôi muốn ủi bộ com_đồ này
6,配 这个 菜 什么 比较 好 ？,Món ăn món này có_thể tốt ?
7,电影院 在 哪 ？,Trạp_chiếu phim ở đâu ?
8,你 今天 几 点 来 ?,Hôm_nay bạn mấy giờ đến ?
9,"不 , 我 会 在 我 逗留 期间 买 的 。","Không , tôi sẽ mua nó ở đâu mua_ ."


## 12) Quick sanity-check

In [13]:
print(submission.head(3))

           tieng_trung                                   tieng_viet
0  还 早 呀 ， 过 几 年 再 说 。  còn sớm mà , qua mấy năm rồi rồi tính nữa .
1             你 跟 我 去吧                           bạn đi tôi tôi nhe
2             我 都 喜欢 。                               Tôi thích nó .
